# Notebook 3 - A First Agent with Agno

**AFDP 2026 - Python Workshop (Block 3, about 20 minutes)**

Everything you did in Notebook 2 - filter, group, average - you can now hand to an **AI agent** and ask in plain English. This notebook builds one in about thirty lines using [Agno](https://docs.agno.com), an open-source Python framework for agents, and Google's Gemini model.

**What is an agent?** A language model on its own can only write text. An *agent* is a language model that has been given **tools** (ordinary Python functions), a set of **instructions**, and permission to loop: read the question, decide which tool to call, look at the result, decide whether it needs another, and finally answer. The arithmetic stays in your Python functions - auditable and deterministic. The model only decides *which* function to call and *how* to explain the result.

That division of labour is the whole point for actuaries: the numbers come from code you can check; the model handles the language.

## 1. Install the libraries

Colab does not have Agno installed, so the first cell installs it. This takes about 30 seconds. (The `-q` keeps the output quiet.)

In [ ]:
!pip install -q agno google-genai
print("Installed.")

## 2. Your Gemini API key

The agent needs a key to talk to Gemini. Google's free tier is enough for this workshop.

1. Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (sign in with your Google account) and click **Create API key**. Copy it.
2. In Colab, click the **key icon** in the left-hand sidebar (*Secrets*), click **Add new secret**, set the name to `GOOGLE_API_KEY`, paste the key as the value, and switch on **Notebook access**.
3. Run the cell below. It reads the secret without ever showing the key on screen.

If you are not in Colab, the cell will ask you to paste the key instead. Never type a key directly into a code cell - notebooks get shared, and keys in shared notebooks get misused.

In [ ]:
import os

try:
    from google.colab import userdata                       # Colab only
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("Key loaded.")

## 3. Load the dataset

Same file, same loading cell as Notebook 2, so the agent works on data you already understand.

In [ ]:
import os
import pandas as pd

FILE_NAME = "us_health_insurance_dataset_afdp.csv"
DATA_URL = "https://raw.githubusercontent.com/rohanyashraj/afdp-python-training/main/2026%20AFDP%20Python%20Training/us_health_insurance_dataset_afdp.csv"   # Direct link to the CSV in the course GitHub repo; set to "" to upload the file manually instead

if DATA_URL:
    insurance_data = pd.read_csv(DATA_URL)
elif os.path.exists(FILE_NAME):
    insurance_data = pd.read_csv(FILE_NAME)
else:
    try:
        from google.colab import files          # only exists inside Google Colab
        print("Please choose", FILE_NAME, "from your computer in the dialog below.")
        files.upload()
        insurance_data = pd.read_csv(FILE_NAME)
    except ImportError:
        raise FileNotFoundError(f"Put {FILE_NAME} in the same folder as this notebook and run this cell again.")

print("Loaded", len(insurance_data), "rows and", insurance_data.shape[1], "columns.")

## 4. A model without tools

First, ask Gemini a question about *our* data with no tools attached. Watch what happens: the model has never seen this file, so it can only guess, decline, or - worst of all - produce a confident number that is wrong. This is why tools matter.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

MODEL_ID = "gemini-3.5-flash-lite"     # free-tier model; change here if Google renames it

plain_model = Agent(model=Gemini(id=MODEL_ID), markdown=True)

plain_model.print_response(
    "In our health insurance dataset, what are the average annual charges for smokers in the southeast region?"
)

## 5. Give the agent tools

A tool is just a Python function with a clear name, typed inputs, and a docstring. The docstring is not decoration here - it is what the model reads to decide when to use the function, so write it as you would explain the function to a colleague.

Each tool below also prints a line when it runs, so you can see the agent working.

In [ ]:
def average_charges(region: str = "all", smoker: str = "all") -> str:
    """Average annual charges in the health insurance dataset, optionally filtered.
    region: 'northeast', 'northwest', 'southeast', 'southwest' or 'all'.
    smoker: 'yes', 'no' or 'all'."""
    print(f"   [tool called] average_charges(region={region!r}, smoker={smoker!r})")
    data = insurance_data
    if region != "all":
        data = data[data["region"] == region.lower()]
    if smoker != "all":
        data = data[data["smoker"] == smoker.lower()]
    if len(data) == 0:
        return "No policyholders match that filter."
    return f"{len(data)} policyholders, average annual charges {data['charges'].mean():,.2f}"


def large_claim_frequency(threshold: float = 20000) -> str:
    """Proportion of policyholders whose annual charges exceed a threshold, split by age band.
    Returns one line per age band with the number of lives and the frequency."""
    print(f"   [tool called] large_claim_frequency(threshold={threshold})")
    bands = pd.cut(insurance_data["age"], bins=[17, 29, 39, 49, 59, 64],
                   labels=["18-29", "30-39", "40-49", "50-59", "60-64"])
    large = insurance_data["charges"] > threshold
    table = large.groupby(bands, observed=True).agg(["count", "mean"])
    lines = [f"{band}: {int(row['count'])} lives, frequency {row['mean']:.1%}" for band, row in table.iterrows()]
    return "\n".join(lines)


# Quick check that the tools work on their own - this is ordinary Python, no AI involved
print(average_charges("southeast", "yes"))
print(large_claim_frequency(20000))

In [ ]:
agent = Agent(
    model=Gemini(id=MODEL_ID),
    tools=[average_charges, large_claim_frequency],
    instructions=[
        "You are an assistant to an actuary analysing a US health insurance dataset.",
        "Always use the tools to get numbers. Never estimate or invent figures.",
        "Quote the numbers the tools return, then add one or two sentences of interpretation.",
    ],
    markdown=True,
)

agent.print_response(
    "What are the average annual charges for smokers in the southeast region, and how does that compare with non-smokers there?",
    stream=True,
)

**What happened.** The model read the question, worked out that it needed `average_charges` twice (once for smokers, once for non-smokers), called your function, and wrote the comparison. The numbers are pandas' numbers - the same ones you would get from Notebook 2 - and the sentences are the model's.

Now a question that needs *both* tools and a little reasoning.

In [ ]:
agent.print_response(
    "Which age band has the highest frequency of large claims above 25,000? "
    "Then tell me whether smoking or age looks like the bigger driver of cost, using the tools.",
    stream=True,
)

## 6. What to take away

- An agent is **model + tools + instructions**, running in a loop. Nothing more mysterious than that.
- The **tools are the actuarial content.** Writing a good tool is writing a good, well-documented Python function - which is what Notebooks 1 and 2 were about.
- The instructions are your **guardrails**: "never invent figures" is a policy, and you wrote it.
- Everything you would want to audit - the filters, the thresholds, the arithmetic - lives in your code, not inside the model.

**Where this goes next.** Give the agent a tool that runs a GLM, one that reads a policy PDF, one that writes an Excel report. Give a *team* of agents different roles (data checker, modeller, reviewer). Same pattern, bigger tools.

## Your turn (if there is time)

1. Add a third tool, `average_bmi(region)`, and ask the agent a question that needs it.
2. Change the instructions so the agent always answers in exactly three bullet points, and see whether it obeys.
3. Ask a question the tools *cannot* answer (for example about motor insurance) and check that the agent says so instead of making something up.

In [ ]:
# Try your own code here